# 02 — Procesamiento de nuevos clientes

Notebook de inferencia para Databricks Jobs. Requiere el modelo registrado versión 1 y el Volume configurado. No reentrena el modelo.

## Configuración

In [ ]:

import os
import mlflow
import mlflow.spark

from pyspark.sql import functions as F
from pyspark.ml.functions import vector_to_array

# Parametros de entrada
dbutils.widgets.text("archivo_csv", "")
dbutils.widgets.text("id_proceso", "")

archivo_csv = dbutils.widgets.get("archivo_csv")
id_proceso = dbutils.widgets.get("id_proceso")

# Configuracion de almacenamiento
ruta_base = "/Volumes/workspace/default/telco_churn"

ruta_entrada = f"{ruta_base}/entradas"
ruta_salida = f"{ruta_base}/resultados"
ruta_temporal = f"{ruta_base}/mlflow_tmp"

# Crear carpetas
os.makedirs(ruta_entrada, exist_ok=True)
os.makedirs(ruta_salida, exist_ok=True)
os.makedirs(ruta_temporal, exist_ok=True)

# Configurar MLflow
mlflow.set_registry_uri("databricks-uc")
os.environ["MLFLOW_DFS_TMP"] = ruta_temporal

# Modelo registrado
ruta_modelo = "models:/workspace.default.telco_churn_model/1"

print("Configuracion inicial correcta")
print("Carpeta de entrada:", ruta_entrada)
print("Carpeta de salida:", ruta_salida)
print("Modelo:", ruta_modelo)


## Lectura y validación

El parámetro `archivo_csv` recibe un nombre de archivo, no una ruta completa. Se requieren cuatro columnas; Churn y TotalCharges son opcionales.

In [ ]:

import uuid

# Modo de prueba con el dataset original
if not archivo_csv:
    ruta_archivo = (
        f"{ruta_base}/WA_Fn-UseC_-Telco-Customer-Churn.csv"
    )
    print("Modo de prueba: dataset original")
else:
    if os.path.basename(archivo_csv) != archivo_csv:
        raise ValueError("Nombre de archivo no valido")

    ruta_archivo = f"{ruta_entrada}/{archivo_csv}"

# Identificador unico de procesamiento
if not id_proceso:
    id_proceso = str(uuid.uuid4())

# Leer archivo CSV
df_nuevo = spark.read.csv(
    ruta_archivo,
    header=True,
    inferSchema=True
)

# Validar columnas indispensables
columnas_requeridas = [
    "customerID",
    "tenure",
    "MonthlyCharges",
    "Contract"
]

columnas_faltantes = [
    c for c in columnas_requeridas
    if c not in df_nuevo.columns
]

if columnas_faltantes:
    raise ValueError(
        f"Columnas faltantes: {columnas_faltantes}"
    )

total_registros = df_nuevo.count()

if total_registros == 0:
    raise ValueError("El archivo no contiene registros")

print("Archivo validado correctamente")
print("ID del proceso:", id_proceso)
print("Total de registros:", total_registros)
print("Total de columnas:", len(df_nuevo.columns))

display(df_nuevo.limit(10))


## Bronze

In [ ]:

# Capa Bronze: datos originales del archivo

df_bronze = (
    df_nuevo
    .withColumn("IdProceso", F.lit(id_proceso))
    .withColumn("FechaCarga", F.current_timestamp())
    .withColumn("ArchivoOrigen", F.lit(os.path.basename(ruta_archivo)))
)

# Ruta independiente para cada proceso
ruta_bronze = f"{ruta_salida}/{id_proceso}/bronze"

# Guardar en formato Delta
df_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .save(ruta_bronze)

print("Capa Bronze creada correctamente")
print("Total de registros:", df_bronze.count())
print("Ruta Bronze:", ruta_bronze)

display(df_bronze.limit(10))


## Silver

In [ ]:

# Capa Silver: limpieza y transformacion

df_silver = spark.read.format("delta").load(ruta_bronze)

# Estandarizar tipos de datos
df_silver = (
    df_silver
    .withColumn("customerID", F.trim(F.col("customerID")))
    .withColumn("Contract", F.trim(F.col("Contract")))
    .withColumn("tenure", F.expr("try_cast(tenure AS INT)"))
    .withColumn(
        "MonthlyCharges",
        F.expr("try_cast(MonthlyCharges AS DOUBLE)")
    )
)

# Limpiar TotalCharges si existe
if "TotalCharges" in df_silver.columns:
    df_silver = df_silver.withColumn(
        "TotalCharges",
        F.expr("try_cast(TotalCharges AS DOUBLE)")
    )

    df_silver = df_silver.withColumn(
        "TotalCharges",
        F.when(
            (F.col("tenure") == 0) &
            F.col("TotalCharges").isNull(),
            0.0
        ).otherwise(F.col("TotalCharges"))
    )

# Convertir abandono real a valor numerico
if "Churn" in df_silver.columns:
    df_silver = df_silver.withColumn(
        "ChurnLabel",
        F.when(F.col("Churn") == "Yes", 1)
         .when(F.col("Churn") == "No", 0)
         .otherwise(F.lit(None).cast("int"))
    )
else:
    df_silver = df_silver.withColumn(
        "ChurnLabel", F.lit(None).cast("int")
    )

# Eliminar registros invalidos
df_silver = df_silver.dropna(
    subset=["customerID", "tenure", "MonthlyCharges", "Contract"]
)

df_silver = df_silver.filter(
    (F.col("customerID") != "") &
    (F.col("tenure") >= 0) &
    (F.col("MonthlyCharges") >= 0) &
    F.col("Contract").isin(
        "Month-to-month", "One year", "Two year"
    )
)

# Eliminar clientes duplicados
df_silver = df_silver.dropDuplicates(["customerID"])

# Validar registros restantes
total_silver = df_silver.count()

if total_silver == 0:
    raise ValueError("No existen registros validos")

# Guardar capa Silver
ruta_silver = f"{ruta_salida}/{id_proceso}/silver"

df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .save(ruta_silver)

print("Capa Silver creada correctamente")
print("Registros originales:", total_registros)
print("Registros limpios:", total_silver)
print("Registros eliminados:", total_registros - total_silver)

display(df_silver.limit(10))


## Gold

In [ ]:

# Capa Gold: preparacion para Machine Learning

df_gold = spark.read.format("delta").load(ruta_silver)

df_gold = df_gold.select(
    "customerID",
    "tenure",
    "Contract",
    "MonthlyCharges",
    "ChurnLabel",
    "IdProceso",
    "FechaCarga",
    "ArchivoOrigen"
)

# Guardar capa Gold
ruta_gold = f"{ruta_salida}/{id_proceso}/gold"

df_gold.write \
    .format("delta") \
    .mode("overwrite") \
    .save(ruta_gold)

print("Capa Gold creada correctamente")
print("Total de registros:", df_gold.count())
print("Total de columnas:", len(df_gold.columns))
print("Ruta Gold:", ruta_gold)

display(df_gold.limit(10))


## Recuperación del modelo

In [ ]:

# Cargar datos de la capa Gold
datos_prediccion = spark.read.format("delta").load(ruta_gold)

# Cargar modelo registrado en Unity Catalog
modelo_cargado = mlflow.spark.load_model(
    ruta_modelo,
    dfs_tmpdir=ruta_temporal
)

# Verificar modelo cargado
print("Modelo cargado correctamente")
print("Modelo: telco_churn_model")
print("Version: 1")
print("Algoritmo: Regresion Logistica")
print("Registros disponibles:", datos_prediccion.count())

# Mostrar etapas del modelo
for etapa in modelo_cargado.stages:
    print(type(etapa).__name__)


## Inferencia

En este notebook: bajo < 30%, medio desde 30%, alto desde 70%. Los umbrales difieren del notebook 01.

In [ ]:

# Ejecutar el modelo sobre los datos cargados
predicciones = modelo_cargado.transform(datos_prediccion)

# Obtener probabilidad de abandono
resultados = predicciones.withColumn(
    "ProbabilidadAbandono",
    F.round(vector_to_array("probability")[1] * 100, 2)
)

# Convertir prediccion numerica a texto
resultados = resultados.withColumn(
    "Prediccion",
    F.when(F.col("prediction") == 1, "Abandona")
     .otherwise("Permanece")
)

# Clasificar nivel de riesgo
resultados = resultados.withColumn(
    "NivelRiesgo",
    F.when(F.col("ProbabilidadAbandono") >= 70, "Alto")
     .when(F.col("ProbabilidadAbandono") >= 30, "Medio")
     .otherwise("Bajo")
)

# Seleccionar columnas para la aplicacion
resultados = resultados.select(
    "customerID",
    "tenure",
    "Contract",
    "MonthlyCharges",
    "ChurnLabel",
    "Prediccion",
    "ProbabilidadAbandono",
    "NivelRiesgo",
    "IdProceso",
    "FechaCarga",
    "ArchivoOrigen"
)

print("Predicciones generadas correctamente")
print("Total de clientes:", resultados.count())

# Mostrar los 20 clientes con mayor riesgo
display(
    resultados.orderBy(
        F.desc("ProbabilidadAbandono")
    ).limit(20)
)


## Exportación y resumen

In [ ]:

# Definir rutas de almacenamiento
ruta_predicciones = f"{ruta_salida}/{id_proceso}/predicciones"
ruta_csv = f"{ruta_salida}/{id_proceso}/csv"

# Guardar predicciones en formato Delta
resultados.write \
    .format("delta") \
    .mode("overwrite") \
    .save(ruta_predicciones)

# Guardar predicciones en formato CSV
resultados.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(ruta_csv)

# Generar resumen de resultados
resumen = resultados.agg(
    F.count("*").alias("TotalClientes"),

    F.sum(
        F.when(F.col("Prediccion") == "Abandona", 1)
         .otherwise(0)
    ).alias("ClientesAbandono"),

    F.sum(
        F.when(F.col("Prediccion") == "Permanece", 1)
         .otherwise(0)
    ).alias("ClientesPermanecen"),

    F.round(
        F.avg("ProbabilidadAbandono"), 2
    ).alias("ProbabilidadPromedio")
)

# Guardar resumen para la aplicacion
ruta_resumen = f"{ruta_salida}/{id_proceso}/resumen"

resumen.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(ruta_resumen)

print("PIPELINE FINALIZADO CORRECTAMENTE")
print("ID del proceso:", id_proceso)
print("Ruta de predicciones:", ruta_predicciones)
print("Ruta CSV:", ruta_csv)
print("Ruta del resumen:", ruta_resumen)

display(resumen)
